# ROS05 · tf2 transform tree and a URDF arm

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/03_ros2/ROS05_tf2_transform_tree_and_a_urdf_arm.ipynb)

**Track:** ROS 2 · **Skill axis (sheet):** Estimation and perception · **Time:** about 65 min

**Skill Hub concepts:** tf2 transform tree; URDF and robot_state_publisher

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each node task has an optional C++ (rclcpp) cell after it. The grader builds it with colcon and runs the same checker against it.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("ROS05")

## Map: frames are a tree of timestamped homogeneous transforms

```
  map ──(localization, jumps allowed)──> odom ──(wheel/visual odometry, smooth, drifts)──> base_link
                                                                                             ├── camera_link  (static)
                                                                                             └── shoulder ─> link1 ─> elbow ─> link2 ─> tool0
                                                                                                 (robot_state_publisher from URDF + /joint_states)
```

| Piece | Publishes | Topic |
|---|---|---|
| `StaticTransformBroadcaster` | fixed mounts (sensor on the chassis), once, latched | `/tf_static` |
| `TransformBroadcaster` | moving frames, continuously, with timestamps | `/tf` |
| `robot_state_publisher` | every link of the URDF from `/joint_states` | `/tf`, `/tf_static`, `/robot_description` |
| `tf2_ros.Buffer` + `TransformListener` | stores the tree, answers `lookup_transform(target, source, time)` | |

Chaining is matrix multiplication: ${}^{odom}T_{cam} = {}^{odom}T_{base}\;{}^{base}T_{cam}$.

Quaternion for a rotation $\theta$ about a unit axis $\hat{u}$:
$q = (\hat{u}\sin\tfrac{\theta}{2},\ \cos\tfrac{\theta}{2})$, so yaw only: $q = (0, 0, \sin\tfrac{\psi}{2}, \cos\tfrac{\psi}{2})$.

URDF: `<link>` elements joined by `<joint type="revolute|prismatic|fixed|continuous">` with `<parent>`, `<child>`,
`<origin xyz rpy>` (pose of the child frame in the parent at joint value 0) and `<axis>`.

In [ ]:
from roboprep import ros

## Task 1: static and moving frames
Node `tf_demo`. Static: `base_link` to `camera_link`, translation (0.1, 0, 0.3), pitched +30 degrees about y.
Dynamic at 20 Hz: `odom` to `base_link` driving a 1 m circle counterclockwise at 0.5 rad/s:
$x = \cos 0.5t,\ y = \sin 0.5t,\ \psi = 0.5t + \pi/2$ (facing along the path).

In [ ]:
%%writefile tf_demo.py
import math
import rclpy
from geometry_msgs.msg import TransformStamped
from rclpy.executors import ExternalShutdownException
from rclpy.node import Node
from tf2_ros import StaticTransformBroadcaster, TransformBroadcaster


def make_tf(stamp, parent, child, xyz, quat):
    t = TransformStamped()
    t.header.stamp, t.header.frame_id, t.child_frame_id = stamp, parent, child
    t.transform.translation.x, t.transform.translation.y, t.transform.translation.z = xyz
    t.transform.rotation.x, t.transform.rotation.y, t.transform.rotation.z, t.transform.rotation.w = quat
    return t


class TfDemo(Node):
    def __init__(self):
        super().__init__("tf_demo")
        self.static = StaticTransformBroadcaster(self)
        self.dynamic = TransformBroadcaster(self)
        self.t0 = self.get_clock().now()
        pass  # TODO: send the static transform once, start a 20 Hz timer

    def tick(self):
        pass  # TODO


def main():
    rclpy.init()
    try:
        rclpy.spin(TfDemo())
    except (KeyboardInterrupt, ExternalShutdownException):
        pass


if __name__ == "__main__":
    main()

In [ ]:
lab.check("tf")

In [ ]:
%%writefile tf_demo.cpp
// C++ (rclcpp) version of Task 1 (optional, same checker). Grade it with lab.check("tf", "tf_demo.cpp"):
// the grader builds it with colcon as package rdl_tf_demo, executable tf_demo (depends on rclcpp, tf2_ros, geometry_msgs).
#include <chrono>
#include <cmath>
#include <geometry_msgs/msg/transform_stamped.hpp>
#include <rclcpp/rclcpp.hpp>
#include <tf2_ros/static_transform_broadcaster.h>
#include <tf2_ros/transform_broadcaster.h>
using namespace std::chrono_literals;

class TfDemo : public rclcpp::Node {
public:
    TfDemo() : Node("tf_demo"), t0_(now()) {
        static_ = std::make_shared<tf2_ros::StaticTransformBroadcaster>(this);
        dynamic_ = std::make_shared<tf2_ros::TransformBroadcaster>(this);
        // TODO: send base_link -> camera_link once (0.1, 0, 0.3), pitched 30 degrees about y;
        //       a 20 Hz timer sending odom -> base_link: x = cos(0.5 t), y = sin(0.5 t), yaw = 0.5 t + pi/2
    }
private:
    rclcpp::Time t0_;
    std::shared_ptr<tf2_ros::StaticTransformBroadcaster> static_;
    std::shared_ptr<tf2_ros::TransformBroadcaster> dynamic_;
};

int main(int argc, char **argv) {
    rclcpp::init(argc, argv);
    rclcpp::spin(std::make_shared<TfDemo>());
    rclcpp::shutdown();
}

In [ ]:
# lab.check("tf", "tf_demo.cpp")  # uncomment after writing the C++ cell above

## Task 2: a two-link arm in URDF
`base_link` -> `shoulder` (revolute about z, origin z = 0.1) -> `link1` -> `elbow` (revolute about z, origin x = 0.3)
-> `link2` -> `tool_joint` (fixed, origin x = 0.25) -> `tool0`. Then publish `/joint_states` with shoulder = 0.5 rad
and elbow = -0.3 rad at 10 Hz.

In [ ]:
%%writefile arm.urdf
<?xml version="1.0"?>
<robot name="two_link_arm">
  <link name="base_link"/>
  <link name="link1"/>
  <link name="link2"/>
  <link name="tool0"/>
  <joint name="shoulder" type="revolute">
    <parent link="base_link"/><child link="link1"/>
    <origin xyz="0 0 0.1" rpy="0 0 0"/><axis xyz="0 0 1"/>
    <limit lower="-3.14" upper="3.14" effort="10" velocity="2"/>
  </joint>
  <!-- TODO: elbow (revolute) and tool_joint (fixed) -->
</robot>

In [ ]:
%%writefile joint_pub.py
import rclpy
from rclpy.executors import ExternalShutdownException
from rclpy.node import Node
from sensor_msgs.msg import JointState


class JointPub(Node):
    def __init__(self):
        super().__init__("joint_pub")
        self.pub = self.create_publisher(JointState, "/joint_states", 10)
        self.create_timer(0.1, self.tick)

    def tick(self):
        pass  # TODO: JointState with a stamp, names and positions


def main():
    rclpy.init()
    try:
        rclpy.spin(JointPub())
    except (KeyboardInterrupt, ExternalShutdownException):
        pass


if __name__ == "__main__":
    main()

In [ ]:
import time
ros.start('ros2 run robot_state_publisher robot_state_publisher --ros-args -p robot_description:="$(cat arm.urdf)"', log="rsp.log")
ros.start("python joint_pub.py")
time.sleep(3)
ros.ros("timeout 4 ros2 run tf2_ros tf2_echo base_link tool0 2>&1 | grep -A2 Translation | head -3", timeout=10)
ros.stop_all()

In [ ]:
lab.check("urdf")

In [ ]:
%%writefile joint_pub.cpp
// C++ (rclcpp) version of Task 2 (optional, same checker). Grade it with lab.check("urdf", "joint_pub.cpp"):
// the grader builds it with colcon as package rdl_joint_pub, executable joint_pub (depends on rclcpp, sensor_msgs).
#include <chrono>
#include <rclcpp/rclcpp.hpp>
#include <sensor_msgs/msg/joint_state.hpp>
using namespace std::chrono_literals;

class JointPub : public rclcpp::Node {
public:
    JointPub() : Node("joint_pub") {
        pub_ = create_publisher<sensor_msgs::msg::JointState>("/joint_states", 10);
        timer_ = create_wall_timer(100ms, [this] {
            // TODO: JointState with a stamp, names {"shoulder", "elbow"}, positions {0.5, -0.3}
        });
    }
private:
    rclcpp::Publisher<sensor_msgs::msg::JointState>::SharedPtr pub_;
    rclcpp::TimerBase::SharedPtr timer_;
};

int main(int argc, char **argv) {
    rclcpp::init(argc, argv);
    rclcpp::spin(std::make_shared<JointPub>());
    rclcpp::shutdown();
}

In [ ]:
# lab.check("urdf", "joint_pub.cpp")  # uncomment after writing the C++ cell above

## Checkpoint: Math
**M1 (Apply).** Shoulder 0, elbow +90 degrees. Distance from the shoulder axis to `tool0` in the xy-plane (meters)?

**M2 (Apply).** The `w` component of the camera's 30-degree pitch quaternion?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Apply).** `lookup_transform("odom", "camera_link", node.get_clock().now())` sometimes throws
"extrapolation into the future". Simplest fix for "latest available"?
- a) add a sleep
- b) pass `rclpy.time.Time()` (time zero), which means "latest common time", or wait with a timeout
- c) publish the static transform faster

**K2 (Understand).** Explain the roles of `map`, `odom` and `base_link` (REP 105): which transform is smooth, which may
jump, and who publishes each.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `ros05.h`).
3. In `roboprep/checks/ros05.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()